# Streaming ESA Project Results Repository (PRR) Data with JupyterGIS

**Authors:** Anne Fouilloux

**Affiliation:** Lifewatch ERIC, Seville, Spain

**Date:** December 2025  

**ESA Contract:** 4000144740/24/I-DT-bgh

---

## Purpose

This notebook demonstrates how to use **JupyterGIS** to discover, stream, and visualize Cloud-Optimized GeoTIFF (COG) data from the [ESA EOP-S Project Results Repository (PRR)](https://eoresults.esa.int/prr_faq.html) — an open data repository for value-added Earth Observation products generated in ESA projects.

## Highlights

- 🔍 **STAC Discovery**: Query the ESA PRR STAC catalog programmatically using `pystac-client`
- 🌍 **COG Streaming**: Stream Cloud-Optimized GeoTIFFs directly without downloading entire files
- 🎨 **Custom Styling**: Apply colorblind-friendly color ramps for scientific visualization
- 🗺️ **Interactive Mapping**: Combine satellite data with OpenStreetMap basemaps
- 💾 **Reproducible Workflow**: Save results as `.jGIS` files for collaboration

<!-- Logos Banner -->
<div style="display: flex; align-items: center; justify-content: space-between; padding: 50px 0; border-bottom: 2px solid #003399;">
    <div style="display: flex; align-items: center;">
        <img src="https://raw.githubusercontent.com/geojupyter/jupytergis/main/packages/base/style/icons/logo.svg" alt="JupyterGIS" width="40" style="margin-right: 80px;">
        <img src="https://brand.esa.int/files/2020/05/ESA_logo_2020_Deep-scaled.jpg" alt="ESA" width="150">
    </div>
    <div style="display: flex; align-items: center;">
        <img src="https://quantstack.net/img/quantstack/logo-website.svg" alt="QuantStack" width="150" style="margin-right: 80px;">
        <img src="https://raw.githubusercontent.com/annefou/jupytergis-showcases/refs/heads/main/content/simula-logo.svg" alt="Simula" width="150" style="margin-right: 80px;">
        <img src="https://www.lifewatch.eu/wp-content/uploads/2021/07/logoLW_eric_outline2-01.svg" alt="Lifewatch ERIC" width="150">
    </div>
</div>

<!-- Badges -->
<p align="center">
    <a href="https://usegalaxy.eu/?tool_id=interactive_tool_jupytergis_notebook&version=latest"><img src="https://img.shields.io/badge/launch-Galaxy%20Europe-green?logo=jupyter" alt="Galaxy Europe"></a>
    <a href="https://github.com/geojupyter/jupytergis"><img src="https://img.shields.io/github/stars/geojupyter/jupytergis?style=social" alt="GitHub Stars"></a>
    <a href="https://jupytergis.readthedocs.io/"><img src="https://img.shields.io/badge/docs-readthedocs-blue" alt="Documentation"></a>
    <a href="https://opensource.org/licenses/BSD-3-Clause"><img src="https://img.shields.io/badge/License-BSD%203--Clause-blue.svg" alt="License"></a>
</p>

## Context

The **ESA Project Results Repository (PRR)** hosts open Earth Observation datasets in analysis-ready formats (COG/ZARR). All data is accessible via a STAC (SpatioTemporal Asset Catalog) API, enabling:

- Programmatic data discovery and search
- Direct streaming of cloud-optimized rasters
- Integration with standard geospatial tools

In this demo, we visualize **ECOSTRESS Land Surface Temperature** data — a product showing mean surface temperatures across Europe derived from the ECOSTRESS mission.

> 📚 **Learn more**: [ESA PRR FAQ](https://eoresults.esa.int/prr_faq.html) | [STAC Browser](https://browser.apex.esa.int/external/eoresults.esa.int/stac)

## Setup

First, we install and import the required libraries.

In [1]:
# Install pystac-client
!pip install pystac-client -q

In [2]:
import os
from pystac_client import Client
from jupytergis import GISDocument

In [3]:
# Add ESA domains to the exempt list
current = os.environ.get("JGIS_EXEMPT_DOMAINS", "")
esa_domains = "https://eoresults.esa.int,https://browser.apex.esa.int"

if current:
    os.environ["JGIS_EXEMPT_DOMAINS"] = f"{current},{esa_domains}"
else:
    # Set with defaults + ESA
    os.environ["JGIS_EXEMPT_DOMAINS"] = f"https://geodes.cnes.fr,https://gdh-portal-prod.cnes.fr,https://geodes-portal.cnes.fr/api/stac/,{esa_domains}"

print("Updated JGIS_EXEMPT_DOMAINS:", os.environ.get("JGIS_EXEMPT_DOMAINS"))

Updated JGIS_EXEMPT_DOMAINS: https://geodes.cnes.fr,https://gdh-portal-prod.cnes.fr,https://geodes-portal.cnes.fr/api/stac/,https://eoresults.esa.int,https://browser.apex.esa.int


## 1. Discover Data from ESA PRR

We connect to the ESA PRR STAC catalog and search for available ECOSTRESS Land Surface Temperature datasets.

In [4]:
# Connect to ESA PRR STAC catalog
catalog = Client.open("https://eoresults.esa.int/stac")

# Search for ECOSTRESS LST collection
search = catalog.search(collections=["ECOSTRESS_LST"], max_items=5)
items = list(search.items())

print(f"📡 Found {len(items)} ECOSTRESS datasets:")
for item in items:
    print(f"   • {item.id}")

📡 Found 3 ECOSTRESS datasets:
   • ECOSTRESS_MEAN_LST_2024
   • ECOSTRESS_MEAN_LST_2023
   • ECOSTRESS_MEAN_LST_2022


## 2. Extract COG URL

The STAC catalog returns asset paths. We construct the full URL for streaming.

In [5]:
# Select the most recent item
item = items[0]

# Build the full COG URL
asset_path = item.assets["PRODUCT"].href
cog_url = f"https://eoresults.esa.int{asset_path}"

print(f"🗂️ Selected: {item.id}")
print(f"🔗 COG URL: {cog_url}")

🗂️ Selected: ECOSTRESS_MEAN_LST_2024
🔗 COG URL: https://eoresults.esa.int/d/ECOSTRESS_LST/2024/06/01/ECOSTRESS_MEAN_LST_2024/2024_LST_AT_merged_composite_mean_70m.tif


## 3. Visualize in JupyterGIS

We create an interactive map combining:
- **OpenStreetMap** basemap for geographic context
- **ECOSTRESS LST** layer with a colorblind-friendly **Inferno** color ramp

The color ramp represents temperature values from 0°C (dark) to 40°C (bright yellow).

> 💡 **Tip**: Use the layer panel on the left to toggle visibility and adjust opacity.

In [6]:
# Create JupyterGIS document centered on Europe
doc = GISDocument("ESA_PRR_Demo.jGIS", latitude=47.5, longitude=14.0, zoom=5)

# Add OpenStreetMap basemap
doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap",
    attribution="© OpenStreetMap contributors"
)

# Create Inferno color ramp (colorblind-friendly)
color = doc.create_color_expr(
    interpolation_type="linear",
    band=1,
    color_stops={
        0.0: [0, 0, 4, 1.0],          # Black (cold)
        0.25: [87, 16, 110, 1.0],     # Purple
        0.5: [188, 55, 84, 1.0],      # Pink-red
        0.75: [249, 142, 9, 1.0],     # Orange
        1.0: [252, 255, 164, 1.0],    # Pale yellow (hot)
    }
)

# Add ECOSTRESS LST layer
doc.add_tiff_layer(
    url=cog_url,
    name=f"ECOSTRESS Land Surface Temperature",
    min=0,
    max=40,
    color_expr=color,
    opacity=0.7
)

# Display the map
doc

## Summary

In this notebook, we demonstrated:

| Step | Description |
|------|-------------|
| **STAC Discovery** | Connected to ESA PRR catalog and searched for datasets |
| **COG Streaming** | Extracted asset URL for direct cloud-optimized access |
| **Visualization** | Created an interactive map with JupyterGIS |
| **Styling** | Applied a colorblind-friendly Inferno color ramp |

The workflow is saved to `ESA_PRR_Demo.jGIS` and can be:
- Reopened in JupyterGIS for further editing
- Shared with collaborators for real-time collaboration
- Exported to QGIS format (`.qgz`)

---

## Next Steps

- 🖊️ **Add annotations**: Right-click on the map to add collaborative annotations
- 👥 **Invite collaborators**: Share the Galaxy Europe URL for real-time editing
- 📊 **Explore other datasets**: Try `worldsoils-soc`, `FCM-AGB-100m`, or other PRR collections

## References & Resources

**Data Sources:**
- [ESA Project Results Repository (PRR)](https://eoresults.esa.int/)
- [ECOSTRESS Mission](https://ecostress.jpl.nasa.gov/)

**Software:**
- [JupyterGIS Documentation](https://jupytergis.readthedocs.io/)
- [pystac-client](https://pystac-client.readthedocs.io/)

**Deployment:**
- This demo runs on [Galaxy Europe](https://usegalaxy.eu/) with full real-time collaboration support

---

*ESA Contract 4000144740/24/I-DT-bgh — Real-time collaboration and collaborative editing for GIS workflows with Jupyter and QGIS*